# S04 — Exercises: NumPy

**Python for Applied Physics** | Master in Applied Physics

Work through all exercises in order. Each exercise builds on the previous one.  
The assertions are your automated test suite — all must pass before you move on.

| Symbol | Difficulty |
|--------|------------|
| 🟢 | Accessible — direct application of lecture content |
| 🟡 | Intermediate — requires combining concepts |
| 🔴 | Challenging — open-ended or physics-heavy |

In [ ]:
import numpy as np
import sys
print(f"NumPy {np.__version__}")

---
## Exercise 1 🟢 — Gaussian beam intensity profile

The on-axis intensity of a Gaussian beam at waist ($z=0$) is:

$$I(r) = \frac{2P}{\pi w_0^2} \exp\!\left(-\frac{2r^2}{w_0^2}\right)$$

**Tasks:**
1. Create a radial grid `r` of 1024 points from $-3w_0$ to $+3w_0$.
2. Compute $I(r)$ for a beam with $w_0 = 300\,\mu\text{m}$ and $P = 500\,\text{mW}$.
3. Find the peak intensity `I_peak` and verify it matches the formula.
4. Find the index `i_hw` where $I$ first drops below $I_{\text{peak}}/e^2$ (i.e. $r = w_0$).
5. Compute the integrated power using `np.trapz` and verify it is close to $P$.

In [ ]:
# Parameters
w0 = 300e-6    # m
P  = 0.5       # W

# 1. Radial grid
r = # YOUR CODE HERE

# 2. Intensity profile
I = # YOUR CODE HERE

# 3. Peak intensity
I_peak = # YOUR CODE HERE
I_peak_formula = 2 * P / (np.pi * w0**2)

# 4. First index where I < I_peak / e²
# Hint: use boolean masking on the positive-r half
i_hw = # YOUR CODE HERE

# 5. Integrated power — use np.trapz with the radial coordinate
# Hint: P = 2π ∫ I(r) r dr  (cylindrical symmetry)
r_pos  = # positive half
I_pos  = # corresponding I
P_integrated = # YOUR CODE HERE

# --- Assertions ---
assert I.shape == (1024,), f"Expected shape (1024,), got {I.shape}"
assert np.isclose(I_peak, I_peak_formula, rtol=1e-6), \
    f"Peak intensity mismatch: {I_peak:.4e} vs {I_peak_formula:.4e}"
assert np.isclose(np.abs(r[i_hw]), w0, rtol=0.02), \
    f"1/e² radius wrong: {np.abs(r[i_hw])*1e6:.1f} µm vs {w0*1e6:.1f} µm"
assert np.isclose(P_integrated, P, rtol=0.02), \
    f"Integrated power {P_integrated:.4f} W ≠ {P} W"

print(f"Peak intensity    : {I_peak/1e4:.2f} W/cm²")
print(f"1/e² radius index : {i_hw}  → r = {np.abs(r[i_hw])*1e6:.1f} µm")
print(f"Integrated power  : {P_integrated*1e3:.2f} mW")
print("All assertions passed ✓")

---
## Exercise 2 🟢 — Boolean masking on a beam cross-section

You have a $128 \times 128$ pixel camera image of a beam (simulated below).  
The camera has a pixel size of $10\,\mu\text{m}$.

**Tasks:**
1. Build the 2D intensity array `I2d` using broadcasting (no loops, no meshgrid).
2. Find the number of pixels above the $1/e^2$ threshold.
3. Compute the centroid $(x_c, y_c)$ of the beam using the intensity-weighted mean.
4. Set all pixels below the $1/e^2$ threshold to zero (`I2d_clipped`).

In [ ]:
# Setup
N_px   = 128
px     = 10e-6       # 10 µm pixel size
w_beam = 200e-6      # 200 µm beam radius
I0_2d  = 1.0         # normalised peak

# 1D coordinate axes (centred on detector)
x_px = (np.arange(N_px) - N_px // 2) * px   # m
y_px = (np.arange(N_px) - N_px // 2) * px   # m

# 1. Build I2d via broadcasting (shape must be (128, 128))
I2d = # YOUR CODE HERE

# 2. Pixels above 1/e² threshold
threshold = I0_2d * np.exp(-2)
n_above   = # YOUR CODE HERE

# 3. Intensity-weighted centroid
# Hint: x_c = sum(I * x) / sum(I)  — use broadcasting or np.sum with axis
x_c = # YOUR CODE HERE
y_c = # YOUR CODE HERE

# 4. Clip below threshold
I2d_clipped = # YOUR CODE HERE

# --- Assertions ---
assert I2d.shape == (128, 128), f"Shape error: {I2d.shape}"
assert np.isclose(I2d[64, 64], I0_2d, rtol=1e-3), "Peak not at centre"
assert n_above > 0, "No pixels above threshold?"
assert np.isclose(x_c, 0.0, atol=px), f"Centroid x off: {x_c*1e6:.1f} µm"
assert np.isclose(y_c, 0.0, atol=px), f"Centroid y off: {y_c*1e6:.1f} µm"
assert I2d_clipped.min() == 0.0, "Clipping failed"

print(f"Pixels above 1/e² : {n_above}")
print(f"Centroid           : ({x_c*1e6:.2f}, {y_c*1e6:.2f}) µm")
print(f"Max after clipping : {I2d_clipped.max():.4f}")
print("All assertions passed ✓")

---
## Exercise 3 🟡 — 2D Gaussian field and radial profile extraction

**Tasks:**
1. Build a complex 2D Gaussian field `E2d` on a $256 \times 256$ grid spanning $\pm 3w$ in both dimensions. Use `gaussian_field_2d` from `shared/optics.py` **or** implement it inline.
2. Compute the 2D intensity `I2d = |E2d|²`.
3. Extract the horizontal cross-section through the centre row.
4. Extract the diagonal cross-section using fancy indexing.
5. Verify that both cross-sections peak at the centre and decay as expected.

In [ ]:
N   = 256
w   = 400e-6   # m
E0  = 1.0

x = np.linspace(-3 * w, 3 * w, N)
y = np.linspace(-3 * w, 3 * w, N)

# 1. Complex 2D Gaussian field
E2d = # YOUR CODE HERE

# 2. Intensity
I2d = # YOUR CODE HERE

# 3. Horizontal cross-section (centre row)
I_horiz = # YOUR CODE HERE

# 4. Diagonal cross-section using fancy indexing
diag_idx = np.arange(N)
I_diag   = # YOUR CODE HERE  (hint: I2d[diag_idx, diag_idx])

# 5. Verify
# At the 1/e² radius along x, I should ≈ I_peak * exp(-2)
# Along diagonal, the 1/e² point is at r = w, so x = y = w/sqrt(2)
I_peak_2d = I2d[N//2, N//2]

# Find I at x = w on horizontal cut
idx_w     = np.argmin(np.abs(x - w))
I_at_w    = I_horiz[idx_w]

# --- Assertions ---
assert E2d.shape == (N, N), f"Shape error: {E2d.shape}"
assert np.isclose(I_peak_2d, E0**2, rtol=1e-3), "Peak not at centre"
assert np.isclose(I_at_w / I_peak_2d, np.exp(-2), rtol=0.01), \
    f"1/e² check failed: {I_at_w/I_peak_2d:.4f} vs {np.exp(-2):.4f}"
assert I_diag.shape == (N,), f"Diagonal shape error: {I_diag.shape}"

print(f"Peak intensity : {I_peak_2d:.4f}")
print(f"I(w) / I_peak  : {I_at_w/I_peak_2d:.4f}  (expected {np.exp(-2):.4f})")
print(f"Diagonal max   : {I_diag.max():.4f}")
print("All assertions passed ✓")

---
## Exercise 4 🟡 — Jones matrix chain

Design a polarisation sequence:

1. Start with **horizontal linear polarisation** $\mathbf{E}_{\text{in}} = [1, 0]$.
2. Pass through a **half-wave plate** (fast axis at $22.5°$) → should produce diagonal polarisation.
3. Pass through a **quarter-wave plate** (fast axis at $0°$) → should produce circular polarisation.
4. Pass through a **linear analyser** at $90°$.

**Tasks:**
- Implement the QWP Jones matrix.
- Compute the output field at each stage.
- Verify the intensity (Stokes $S_0$) is conserved through the wave plates.
- Compute the transmission through the analyser.

In [ ]:
def linear_polariser(angle_deg: float) -> np.ndarray:
    θ = np.radians(angle_deg)
    c, s = np.cos(θ), np.sin(θ)
    return np.array([[c**2, c*s], [c*s, s**2]])

def half_wave_plate(fast_axis_deg: float) -> np.ndarray:
    θ = np.radians(fast_axis_deg)
    c, s = np.cos(2*θ), np.sin(2*θ)
    return np.array([[c, s], [s, -c]])

def quarter_wave_plate(fast_axis_deg: float) -> np.ndarray:
    """
    Jones matrix for a quarter-wave plate (retardation = π/2).
    Fast axis at angle θ from x-axis.
    """
    # YOUR CODE HERE
    # Hint: the general retarder matrix is
    #   J = exp(-iδ/2) * [[cos²θ + exp(iδ)sin²θ, (1-exp(iδ))cosθsinθ],
    #                      [(1-exp(iδ))cosθsinθ,  sin²θ + exp(iδ)cos²θ]]
    # For QWP: δ = π/2
    pass

# Input
E_in = np.array([1.0 + 0j, 0.0 + 0j])   # horizontal

# Stage 1: HWP at 22.5°
E1 = # YOUR CODE HERE

# Stage 2: QWP at 0°
E2 = # YOUR CODE HERE

# Stage 3: Analyser at 90°
E3 = # YOUR CODE HERE

# Intensities
I_in = np.sum(np.abs(E_in)**2)
I1   = np.sum(np.abs(E1)**2)
I2   = np.sum(np.abs(E2)**2)
I3   = np.sum(np.abs(E3)**2)

# --- Assertions ---
assert quarter_wave_plate is not None, "Implement quarter_wave_plate first"
assert np.isclose(I1, I_in, atol=1e-10), f"HWP should conserve intensity: {I1}"
assert np.isclose(I2, I_in, atol=1e-10), f"QWP should conserve intensity: {I2}"
# After HWP(22.5°): should be diagonal → |Ex| ≈ |Ey|
assert np.isclose(np.abs(E1[0]), np.abs(E1[1]), atol=1e-6), \
    f"HWP should give diagonal polarisation: {E1}"
# After QWP(0°): should be circular → |Ex| ≈ |Ey|, phase diff ≈ π/2
assert np.isclose(np.abs(E2[0]), np.abs(E2[1]), atol=1e-6), \
    f"Should be circular after QWP: {E2}"

print(f"I_in    : {I_in:.4f}")
print(f"I after HWP : {I1:.4f}")
print(f"I after QWP : {I2:.4f}")
print(f"Phase diff  : {np.angle(E2[1]/E2[0])*180/np.pi:.1f}°  (expected ±90°)")
print(f"Transmission through analyser: {I3:.4f}")
print("All assertions passed ✓")

---
## Exercise 5 🟡 — Pulse spectrum and time–bandwidth product

Use `pulse_spectrum` from `shared/pulses.py` (or implement inline) to:

1. Build a Gaussian pulse with $\tau = 50\,\text{fs}$ on a time grid of $N = 4096$ points, $\delta t = 5\,\text{fs}$.
2. Compute the spectrum.
3. Measure the FWHM in time and frequency.
4. Compute the time–bandwidth product and compare to the transform limit $0.4413$.
5. Repeat with $\tau = 200\,\text{fs}$ and verify that the TBP is the same.

In [ ]:
def pulse_spectrum(t, E_t):
    """Return (freq, S) — centred FFT of E_t."""
    N  = len(t)
    dt = t[1] - t[0]
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
    E_f  = np.fft.fftshift(np.fft.fft(E_t))
    return freq, np.abs(E_f)**2


def fwhm(x, y):
    """Estimate FWHM of a peaked array y defined on axis x."""
    # YOUR CODE HERE
    # Hint: normalise y, find indices where y >= 0.5, take max - min of x there
    pass


# Pulse 1: τ = 50 fs
N   = 4096
dt  = 5e-15
τ1  = 50e-15

t    = (np.arange(N) - N // 2) * dt
E_t1 = # YOUR CODE HERE

freq1, S1 = pulse_spectrum(t, E_t1)

Δt1  = fwhm(t, np.abs(E_t1)**2)
Δν1  = fwhm(freq1, S1)
TBP1 = Δt1 * Δν1

# Pulse 2: τ = 200 fs (same grid)
τ2   = 200e-15
E_t2 = # YOUR CODE HERE
freq2, S2 = pulse_spectrum(t, E_t2)
Δt2  = fwhm(t, np.abs(E_t2)**2)
Δν2  = fwhm(freq2, S2)
TBP2 = Δt2 * Δν2

TBP_limit = 2 * np.log(2) / np.pi

# --- Assertions ---
assert fwhm is not None and Δt1 is not None, "Implement fwhm first"
assert np.isclose(TBP1, TBP_limit, rtol=0.01), \
    f"TBP1 = {TBP1:.4f}, expected {TBP_limit:.4f}"
assert np.isclose(TBP2, TBP_limit, rtol=0.01), \
    f"TBP2 = {TBP2:.4f}, expected {TBP_limit:.4f}"
assert Δt2 > Δt1, "Longer τ should give larger Δt"
assert Δν2 < Δν1, "Longer τ should give narrower spectrum"

print(f"τ = {τ1*1e15:.0f} fs:  Δt={Δt1*1e15:.1f} fs, Δν={Δν1/1e12:.2f} THz, TBP={TBP1:.4f}")
print(f"τ = {τ2*1e15:.0f} fs:  Δt={Δt2*1e15:.1f} fs, Δν={Δν2/1e12:.2f} THz, TBP={TBP2:.4f}")
print(f"Transform limit: {TBP_limit:.4f}")
print("All assertions passed ✓")

---
## Exercise 6 🔴 — Chirped pulse and group delay dispersion

A chirped pulse acquires a quadratic spectral phase:

$$\tilde{E}_{\text{chirped}}(\omega) = \tilde{E}_0(\omega) \cdot e^{i \frac{1}{2} \text{GDD} \cdot \omega^2}$$

where $\omega = 2\pi(\nu - \nu_0)$ is the offset frequency.

**Tasks:**
1. Start with the $\tau = 50\,\text{fs}$ pulse from Exercise 5.
2. Apply GDD = $+500\,\text{fs}^2$ in the frequency domain.
3. IFFT back to time. Compute the new pulse duration (FWHM).
4. Verify that the spectrum $|\tilde{E}(\omega)|^2$ is unchanged.
5. Compute and plot (describe numerically) the **instantaneous frequency**:

$$\nu_{\text{inst}}(t) = \nu_0 - \frac{1}{2\pi} \frac{d\phi(t)}{dt}$$

where $\phi(t) = \arg E(t)$. Use `np.diff` and `np.unwrap`.

6. Show that the chirp rate $d\nu_{\text{inst}}/dt$ matches the expected value from GDD.

In [ ]:
# Parameters (continue from Exercise 5)
N    = 4096
dt   = 5e-15
τ    = 50e-15
λ0   = 800e-9
c    = 3e8
ν0   = c / λ0
GDD  = 500e-30    # 500 fs²

t    = (np.arange(N) - N // 2) * dt
E_t  = np.exp(-t**2 / (2 * τ**2))   # real Gaussian envelope

# Frequency axis
freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
ω    = 2 * np.pi * freq              # angular frequency (offset from DC)

# 1. FFT
E_f = # YOUR CODE HERE  (fft + fftshift)

# 2. Apply GDD phase — note ω here is offset from DC, not from ω0
# For a baseband representation (no carrier), ω0 = 0
phase_GDD   = # YOUR CODE HERE
E_f_chirped = # YOUR CODE HERE

# 3. IFFT back to time
E_t_chirped = # YOUR CODE HERE  (ifftshift before ifft)

# Pulse duration of chirped pulse
I_chirped = np.abs(E_t_chirped)**2
Δt_chirped = fwhm(t, I_chirped)

# Analytical expectation: Δt_chirped = Δt_TL * sqrt(1 + (GDD/τ²)²)
Δt_TL      = 2 * np.sqrt(2 * np.log(2)) * τ   # FWHM of transform-limited pulse
Δt_expected = Δt_TL * np.sqrt(1 + (GDD / τ**2)**2)

# 4. Verify spectrum unchanged
_, S_original = pulse_spectrum(t, E_t)
_, S_chirped  = pulse_spectrum(t, E_t_chirped)

# 5 & 6. Instantaneous frequency
phase_t   = # np.unwrap(np.angle(E_t_chirped))
# dφ/dt via np.diff (result is N-1 points)
dphase_dt = # YOUR CODE HERE
ν_inst    = # ν0 - dphase_dt / (2π)   — skip ν0 since baseband
t_mid     = # midpoints of t (for plotting)

# Chirp rate from linear fit
# For a Gaussian chirped pulse the linear chirp rate is GDD/τ² in appropriate units
# Fit ν_inst vs t_mid in the central region
central = np.abs(t_mid) < 2 * Δt_chirped
coeffs  = np.polyfit(t_mid[central], ν_inst[central], 1)  # linear fit
chirp_rate_measured = coeffs[0]   # Hz/s

# Expected: dν/dt = GDD / (2π τ⁴ + 2π GDD²/τ²)  (simplified for large GDD limit: ≈ 1/(2π·GDD))
chirp_rate_expected = 1 / (2 * np.pi * GDD)   # approximate for GDD >> τ²

# --- Assertions ---
assert np.isclose(Δt_chirped, Δt_expected, rtol=0.05), \
    f"Chirped duration {Δt_chirped*1e15:.1f} fs ≠ {Δt_expected*1e15:.1f} fs"
assert np.allclose(S_original / S_original.max(),
                   S_chirped  / S_chirped.max(), atol=1e-6), \
    "Spectrum changed after GDD — check your FFT normalisation"

print(f"Transform-limited FWHM : {Δt_TL*1e15:.1f} fs")
print(f"Chirped FWHM (measured): {Δt_chirped*1e15:.1f} fs")
print(f"Chirped FWHM (expected): {Δt_expected*1e15:.1f} fs")
print(f"Spectrum preserved: {np.allclose(S_original/S_original.max(), S_chirped/S_chirped.max(), atol=1e-6)}")
print(f"Chirp rate (measured) : {chirp_rate_measured/1e24:.3f} ×10²⁴ Hz/s")
print(f"Chirp rate (expected) : {chirp_rate_expected/1e24:.3f} ×10²⁴ Hz/s")
print("All assertions passed ✓")

---
## Exercise 7 🔴 — Multi-element ABCD beam propagation

Design a beam expander followed by a focusing lens and compute the beam radius at every position along the optical axis.

**System:** input collimated beam $w_0 = 1\,\text{mm}$, $\lambda = 1030\,\text{nm}$

| Element | Position |
|---------|----------|
| Entrance | $z = 0$ |
| Lens 1 ($f_1 = 50\,\text{mm}$) | $z = 50\,\text{mm}$ |
| Lens 2 ($f_2 = 200\,\text{mm}$) | $z = 150\,\text{mm}$ |
| Focusing lens ($f_3 = 100\,\text{mm}$) | $z = 450\,\text{mm}$ |

**Tasks:**
1. Build the `free_space` and `thin_lens` ABCD matrices.
2. Propagate a ray $[w_0, 0]$ through each element using matrix multiplication.
3. Compute the beam radius at 1000 positions along $z \in [0, 600]\,\text{mm}$ by applying the appropriate cumulative system matrix at each $z$.
4. Find the position of minimum beam radius (focus) after the last lens.
5. Compare the focal spot size to $w_f = f_3 \lambda / (\pi w_{\text{in}})$ where $w_{\text{in}}$ is the beam radius just before lens 3.

In [ ]:
λ  = 1030e-9   # m
w0 = 1e-3      # 1 mm input beam radius

def free_space(d: float) -> np.ndarray:
    """ABCD matrix for free-space propagation."""
    # YOUR CODE HERE
    pass

def thin_lens(f: float) -> np.ndarray:
    """ABCD matrix for a thin lens."""
    # YOUR CODE HERE
    pass

# Element positions
z_L1, f1 = 50e-3,  50e-3
z_L2, f2 = 150e-3, 200e-3
z_L3, f3 = 450e-3, 100e-3

# Propagation grid
z = np.linspace(0, 600e-3, 1000)
w = np.zeros_like(z)   # beam radius at each z

# For each z, build the cumulative system matrix and apply to input ray
ray_in = np.array([w0, 0.0])   # collimated: angle = 0

# YOUR CODE HERE
# Strategy: iterate over z. For each z, build M = product of matrices
# from z=0 up to that z (including any lenses in between).
# The beam radius is the |y| component of the output ray.
#
# Hint: define a helper that returns the system matrix for a given z

# Find focus after L3
mask_after_L3 = z > z_L3
z_focus   = # YOUR CODE HERE  (z where w is minimum, after L3)
w_focus   = # YOUR CODE HERE

# Beam radius just before L3
idx_before_L3 = np.argmin(np.abs(z - (z_L3 - 1e-6)))
w_in_L3   = w[idx_before_L3]

# Diffraction-limited spot
w_f_expected = f3 * λ / (np.pi * w_in_L3)

# --- Assertions ---
assert free_space(0.1) is not None, "Implement free_space"
assert thin_lens(0.1) is not None, "Implement thin_lens"
assert w.shape == (1000,), f"w shape error: {w.shape}"
assert np.all(w >= 0), "Beam radius must be non-negative"
assert w_focus < w0, "Focus should be smaller than input beam"
# Focus should be within ~10 mm of the expected focal plane (z_L3 + f3)
assert np.abs(z_focus - (z_L3 + f3)) < 10e-3, \
    f"Focus at {z_focus*1e3:.1f} mm, expected near {(z_L3+f3)*1e3:.1f} mm"

print(f"Beam radius just before L3 : {w_in_L3*1e3:.2f} mm")
print(f"Focus position             : {z_focus*1e3:.1f} mm")
print(f"Focus radius (measured)    : {w_focus*1e6:.1f} µm")
print(f"Focus radius (expected)    : {w_f_expected*1e6:.1f} µm")
print("All assertions passed ✓")

---
## Wrap-up checklist

Before committing your work:

- [ ] All 7 exercises: assertions pass
- [ ] `shared/optics.py` updated with `gaussian_intensity` and `gaussian_field_2d`
- [ ] `shared/pulses.py` updated with `freq_axis` and `pulse_spectrum`
- [ ] Changes committed: `git add shared/ && git commit -m "S04: add beam and FFT helpers to shared modules"`

**Next session:** S05 — Matplotlib: visualising physical fields and pulse data.